# Schema Validation 3 results and analysis

Audit the completed Schema v1.2 run, summarize model screening outputs, and
adjudicate every surfaced candidate. The substantive unit is one snapshot, and
the full-run corpus is the same 202 filenames used in Validation 2.

Model assessments are screening outputs, not accepted schema changes. Calibration
evidence is reported separately. No-gap cases were not audited, so this exercise
does not estimate recall or establish universal completeness. Comparisons with
Validation 2 are descriptive only because the schema, prompt, and response
contract changed; they cannot establish an effect caused by Markdown or Pydantic.

Cost uses recorded GPT-5.5 Flex tokens. The current GPT-5.5 model page lists
Standard rates of **$5.00/M input, $0.50/M cached input, and $30.00/M output**;
OpenAI describes Flex as half the Standard price. This notebook therefore uses
**$2.50/M uncached input, $0.25/M cached input, and $15.00/M output**, checked
September 8, 2026. See the [GPT-5.5 model page](https://developers.openai.com/api/docs/models/gpt-5.5)
and [GPT-5.5 announcement](https://openai.com/index/introducing-gpt-5-5/).


In [ ]:
import json
from collections import Counter
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import Image, Markdown, display

from data_snapshot.constants import ROOT


## Configuration


In [ ]:
VALIDATION_DIR = ROOT / "notebooks/schema_validation3"
OUTPUTS_DIR = VALIDATION_DIR / "outputs"
RESULTS_PATH = OUTPUTS_DIR / "results.jsonl"
ERRORS_PATH = OUTPUTS_DIR / "errors.jsonl"
CALIBRATION_PATHS = {
    "calibration0_complete_schema": OUTPUTS_DIR / "calibration_results0.jsonl",
    "calibration1_provenance_ablation": OUTPUTS_DIR / "calibration_results1.jsonl",
}
CALIBRATION_ERROR_PATHS = {
    "calibration0_complete_schema": OUTPUTS_DIR / "calibration_errors0.jsonl",
    "calibration1_provenance_ablation": OUTPUTS_DIR / "calibration_errors1.jsonl",
}
VALIDATION2_RESULTS_PATH = ROOT / "notebooks/schema_validation2/outputs/results.jsonl"
SNAPSHOTS_DIR = ROOT / "notebooks/schema_validation1/data/snapshots"
ASSESSMENT_ORDER = ["no_critical_gap_found", "possible_gap", "critical_gap_found"]
GAP_STATUS_ORDER = ["possible", "critical"]
PRICES_USD_PER_1M = {
    ("gpt-5.5", "flex"): {
        "input": 2.50,
        "cached_input": 0.25,
        "output": 15.00,
    }
}


In [ ]:
def load_jsonl(path: Path) -> list[dict[str, Any]]:
    """Load JSON objects from a JSONL file.

    Parameters
    ----------
    path : Path
        JSONL file to read.

    Returns
    -------
    list[dict[str, Any]]
        Parsed records, or an empty list if the file does not exist.

    Raises
    ------
    ValueError
        If a non-empty line is not valid JSON.
    """
    if not path.exists():
        return []
    records = []
    with path.open(encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            if not line.strip():
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"Invalid JSON in {path} at line {line_number}."
                ) from exc
    return records


def get_service_tier(record: dict[str, Any]) -> str:
    """Return the recorded service tier.

    Parameters
    ----------
    record : dict[str, Any]
        Validation result record.

    Returns
    -------
    str
        Response tier, or the requested tier if absent.
    """
    raw_response = record.get("raw_response") or {}
    request_config = record.get("request_config") or {}
    return raw_response.get("service_tier") or request_config.get(
        "service_tier", "default"
    )


def calculate_cost_usd(record: dict[str, Any]) -> float:
    """Calculate estimated token cost for one call.

    Parameters
    ----------
    record : dict[str, Any]
        Successful result with usage data.

    Returns
    -------
    float
        Estimated cost in US dollars.

    Raises
    ------
    ValueError
        If usage is absent or cached tokens exceed input tokens.
    KeyError
        If pricing is not configured.
    """
    usage = record.get("usage")
    if not isinstance(usage, dict):
        raise ValueError(f"Missing usage for {record.get('snapshot_file_name')}.")
    prices = PRICES_USD_PER_1M[(record["model"], get_service_tier(record))]
    input_tokens = int(usage["input_tokens"])
    cached_tokens = int(
        (usage.get("input_tokens_details") or {}).get("cached_tokens", 0)
    )
    if cached_tokens > input_tokens:
        raise ValueError("Cached input tokens cannot exceed input tokens.")
    cost = (input_tokens - cached_tokens) * prices["input"]
    cost += cached_tokens * prices["cached_input"]
    cost += int(usage["output_tokens"]) * prices["output"]
    return cost / 1_000_000


def expected_assessment(gaps: list[dict[str, Any]]) -> str:
    """Infer the assessment implied by candidate statuses.

    Parameters
    ----------
    gaps : list[dict[str, Any]]
        Candidate gaps for one snapshot.

    Returns
    -------
    str
        Expected snapshot assessment.
    """
    statuses = {gap["gap_status"] for gap in gaps}
    if "critical" in statuses:
        return "critical_gap_found"
    if statuses:
        return "possible_gap"
    return "no_critical_gap_found"


## Load and normalize the full run


In [ ]:
results = load_jsonl(RESULTS_PATH)
errors = load_jsonl(ERRORS_PATH)
validation2_results = load_jsonl(VALIDATION2_RESULTS_PATH)
if not results or not validation2_results:
    raise ValueError("Validation 2 and Validation 3 results are required.")

expected_files = {row["snapshot_file_name"] for row in validation2_results}
result_names = [row["snapshot_file_name"] for row in results]
successful_files = set(result_names)
duplicate_files = [name for name, count in Counter(result_names).items() if count > 1]
unresolved_errors = [
    error for error in errors if error.get("snapshot_file_name") not in successful_files
]

result_rows = []
gap_rows = []
assessment_mismatches = []
for record in results:
    usage = record["usage"]
    parsed = record["parsed_output"]
    gaps = parsed["critical_or_possible_gaps"]
    assessment = parsed["coverage_assessment"]
    cached_tokens = int(
        (usage.get("input_tokens_details") or {}).get("cached_tokens", 0)
    )
    reasoning_tokens = int(
        (usage.get("output_tokens_details") or {}).get("reasoning_tokens", 0)
    )
    raw_response = record.get("raw_response") or {}
    if assessment != expected_assessment(gaps):
        assessment_mismatches.append(record["snapshot_file_name"])
    result_rows.append(
        {
            "snapshot_file_name": record["snapshot_file_name"],
            "source": record["source"],
            "artifact_type": record["artifact_type"],
            "schema_version": record["schema_version"],
            "excluded_schema_fields": record["excluded_schema_fields"],
            "model": record["model"],
            "raw_model": raw_response.get("model"),
            "api_status": record["api_status"],
            "service_tier": get_service_tier(record),
            "coverage_assessment": assessment,
            "candidate_gaps": len(gaps),
            "input_tokens": int(usage["input_tokens"]),
            "cached_input_tokens": cached_tokens,
            "uncached_input_tokens": int(usage["input_tokens"]) - cached_tokens,
            "output_tokens": int(usage["output_tokens"]),
            "reasoning_tokens": reasoning_tokens,
            "total_tokens": int(usage["total_tokens"]),
            "elapsed_seconds": float(record["elapsed_seconds"]),
            "cost_usd": calculate_cost_usd(record),
        }
    )
    context = {
        "snapshot_file_name": record["snapshot_file_name"],
        "source": record["source"],
        "artifact_type": record["artifact_type"],
        "source_document_id": record["source_document_id"],
    }
    for gap in gaps:
        gap_rows.append(context | gap)

results_df = pd.DataFrame(result_rows)
records_by_name = {row["snapshot_file_name"]: row for row in results}
gap_columns = [
    "snapshot_file_name",
    "source",
    "artifact_type",
    "source_document_id",
    "gap_id",
    "gap_status",
    "missing_or_inadequately_represented_metadata",
    "evidence",
    "faithful_representation",
    "why_snapshot_metadata",
    "closest_schema_paths",
    "why_existing_schema_may_be_insufficient",
    "material_impact",
    "material_consequence",
    "uncertainty_note",
]
gaps_df = pd.DataFrame(gap_rows, columns=gap_columns)
print(
    f"Loaded {len(results):,} results, {len(errors):,} error attempts, "
    f"and {len(gaps_df):,} candidate gaps."
)


## Run integrity


In [ ]:
request_config_counts = Counter(
    json.dumps(row["request_config"], sort_keys=True) for row in results
)
raw_model_counts = Counter(
    (row.get("raw_response") or {}).get("model") for row in results
)
api_status_counts = Counter(row["api_status"] for row in results)
schema_version_counts = Counter(row["schema_version"] for row in results)
excluded_field_counts = Counter(tuple(row["excluded_schema_fields"]) for row in results)
integrity_rows = [
    {
        "check": "Same filename set as Validation 2",
        "observed": f"{len(successful_files)} / {len(expected_files)}",
        "status": (
            "PASS"
            if (
                successful_files == expected_files
                and len(results) == len(expected_files)
            )
            else "REVIEW"
        ),
    },
    {
        "check": "No duplicate filenames",
        "observed": len(duplicate_files),
        "status": "PASS" if not duplicate_files else "REVIEW",
    },
    {
        "check": "No unresolved errors",
        "observed": len(unresolved_errors),
        "status": "PASS" if not unresolved_errors else "REVIEW",
    },
    {
        "check": "All calls completed",
        "observed": dict(api_status_counts),
        "status": (
            "PASS" if (api_status_counts == {"completed": len(results)}) else "REVIEW"
        ),
    },
    {
        "check": "Assessment matches statuses",
        "observed": len(assessment_mismatches),
        "status": "PASS" if not assessment_mismatches else "REVIEW",
    },
    {
        "check": "Complete v1.2 schema used",
        "observed": {
            "versions": dict(schema_version_counts),
            "excluded": {str(k): v for k, v in excluded_field_counts.items()},
        },
        "status": (
            "PASS"
            if (
                schema_version_counts == {"1.2": len(results)}
                and excluded_field_counts == {(): len(results)}
            )
            else "REVIEW"
        ),
    },
    {
        "check": "One request configuration",
        "observed": len(request_config_counts),
        "status": "PASS" if len(request_config_counts) == 1 else "REVIEW",
    },
    {
        "check": "One resolved model",
        "observed": dict(raw_model_counts),
        "status": "PASS" if len(raw_model_counts) == 1 else "REVIEW",
    },
]
integrity_df = pd.DataFrame(integrity_rows)
display(integrity_df.style.hide(axis="index"))
assert (integrity_df["status"] == "PASS").all()
display(Markdown("**Integrity result: all checks passed.**"))


In [ ]:
config = json.loads(next(iter(request_config_counts)))
display(
    pd.DataFrame(
        [
            {
                "calls": len(results),
                "model_alias": results[0]["model"],
                "resolved_model": results[0]["raw_response"]["model"],
                "service_tier": config["service_tier"],
                "reasoning_effort": config["reasoning"]["effort"],
                "max_output_tokens": config["max_output_tokens"],
                "prompt_cache_key": config["prompt_cache_key"],
                "schema_version": results[0]["schema_version"],
            }
        ]
    ).style.hide(axis="index")
)


## API spend and runtime


In [ ]:
overall_cost = pd.DataFrame(
    {
        "successful_calls": [len(results_df)],
        "total_api_spend_usd": [results_df["cost_usd"].sum()],
        "average_cost_per_call_usd": [results_df["cost_usd"].mean()],
        "input_tokens": [results_df["input_tokens"].sum()],
        "cached_input_tokens": [results_df["cached_input_tokens"].sum()],
        "uncached_input_tokens": [results_df["uncached_input_tokens"].sum()],
        "output_tokens": [results_df["output_tokens"].sum()],
        "reasoning_tokens": [results_df["reasoning_tokens"].sum()],
        "summed_elapsed_minutes": [results_df["elapsed_seconds"].sum() / 60],
    }
)
display(
    overall_cost.style.format(
        {
            "total_api_spend_usd": "$ {:,.4f}",
            "average_cost_per_call_usd": "$ {:,.4f}",
            "input_tokens": "{:,}",
            "cached_input_tokens": "{:,}",
            "uncached_input_tokens": "{:,}",
            "output_tokens": "{:,}",
            "reasoning_tokens": "{:,}",
            "summed_elapsed_minutes": "{:,.1f}",
        }
    ).hide(axis="index")
)


## Full-run screening summary


In [ ]:
assessment_summary = (
    results_df["coverage_assessment"]
    .value_counts()
    .reindex(ASSESSMENT_ORDER, fill_value=0)
    .rename_axis("coverage_assessment")
    .reset_index(name="snapshots")
)
assessment_summary["percentage"] = (
    100 * assessment_summary["snapshots"] / len(results_df)
)
display(
    assessment_summary.style.format(
        {"snapshots": "{:,}", "percentage": "{:.1f}%"}
    ).hide(axis="index")
)
display(
    Markdown(
        f"""The model returned **{len(gaps_df)} candidates** across
**{gaps_df["snapshot_file_name"].nunique()} snapshots**. These are
pre-adjudication screening counts."""
    )
)


In [ ]:
def assessment_breakdown(column: str) -> pd.DataFrame:
    """Build an assessment cross-tabulation.

    Parameters
    ----------
    column : str
        Result column used to group snapshots.

    Returns
    -------
    pandas.DataFrame
        Assessment counts, totals, and flagged percentages.
    """
    table = pd.crosstab(results_df[column], results_df["coverage_assessment"]).reindex(
        columns=ASSESSMENT_ORDER, fill_value=0
    )
    table["total"] = table.sum(axis=1)
    table["flagged_percentage"] = (
        100 * (table["possible_gap"] + table["critical_gap_found"]) / table["total"]
    )
    return table


display(assessment_breakdown("source"))
display(assessment_breakdown("artifact_type"))


## Descriptive comparison with Validation 2

The filenames are identical, but the schema and instrument differ. Assessment
changes are therefore not regressions and cannot be attributed to Markdown or
Pydantic.


In [ ]:
validation2_assessments = {
    row["snapshot_file_name"]: row["parsed_output"]["coverage_assessment"]
    for row in validation2_results
}
comparison_df = results_df[["snapshot_file_name", "coverage_assessment"]].rename(
    columns={"coverage_assessment": "validation3"}
)
comparison_df["validation2"] = comparison_df["snapshot_file_name"].map(
    validation2_assessments
)
display(pd.crosstab(comparison_df["validation2"], comparison_df["validation3"]))


## Candidate-gap review


In [ ]:
gaps_for_display = gaps_df.copy()
gaps_for_display["closest_schema_paths"] = gaps_for_display[
    "closest_schema_paths"
].str.join(", ")
gap_status_summary = (
    gaps_df["gap_status"]
    .value_counts()
    .reindex(GAP_STATUS_ORDER, fill_value=0)
    .rename_axis("gap_status")
    .reset_index(name="candidate_rows")
)
impact_summary = (
    gaps_df["material_impact"]
    .value_counts()
    .rename_axis("material_impact")
    .reset_index(name="candidate_rows")
)
closest_path_summary = (
    pd.Series(
        [path for row in gaps_df.itertuples() for path in row.closest_schema_paths],
        name="closest_schema_path",
    )
    .value_counts()
    .rename_axis("closest_schema_path")
    .reset_index(name="candidate_mentions")
)
display(gap_status_summary.style.hide(axis="index"))
display(impact_summary.style.hide(axis="index"))
display(closest_path_summary.style.hide(axis="index"))


## Candidate-by-candidate analyst adjudication

Every surfaced candidate is assigned to one family. This is a first-pass
analyst decision for review, not an automated relabeling of model severity. The
explicit filename and gap-ID mapping fails if the frozen outputs change.


In [ ]:
FAMILY_REVIEW = {
    "panel": (
        "retain_representation_issue",
        "Panel-specific metadata cannot be attached without flattening.",
    ),
    "hierarchy": (
        "retain_representation_issue",
        "One-level category_groups cannot preserve deeper hierarchy.",
    ),
    "scoped": (
        "retain_representation_issue",
        "A value exists, but its variable, subset, or component scope is lost.",
    ),
    "covered": (
        "covered_by_v1_2",
        "Existing fields, source text, or separate records are adequate.",
    ),
    "visual": (
        "out_of_scope_visual_detail",
        "Visual-encoding mechanics are below the selected granularity.",
    ),
    "cartography": (
        "out_of_scope_cartography",
        "Map scale, coordinates, and extent remain outside scope.",
    ),
    "crossref": (
        "document_relationship_not_new_snapshot_field",
        "Cross-snapshot references belong in notes or document context.",
    ),
    "operational": (
        "operational_or_extraction_content",
        "The content is operational rather than descriptive metadata.",
    ),
    "insufficient": (
        "insufficient_evidence_or_materiality",
        "The evidence does not establish a material schema limitation.",
    ),
}


In [ ]:
ADJUDICATION_TSV = """document_10455617_figure_000.png	G1_variable_specific_temporal_and_reference_years	scoped
document_11777361_figure_006.png	G1	visual
document_11867889_figure_001.png	gap_001_panel_scoped_metadata_for_composite_figure	panel
document_11867889_figure_004.png	G1	covered
document_12916918_figure_014.png	gap_001_panel_variable_method_relationships	panel
document_13840629_figure_004.png	gap_001_dimension_visual_encoding_roles	visual
document_14861148_figure_018.png	gap_001_dual_axis_side_assignment	visual
document_15347644_figure_000.png	G1	panel
document_15348558_figure_006.png	panel_level_variable_axis_mapping	panel
document_8772744_figure_010.png	G1	visual
document_11017288_table_005.png	gap_001_cross_table_footnote_dependency	crossref
document_11174028_table_006.png	G1_nested_table_header_and_row_group_hierarchy	hierarchy
document_11174028_table_006.png	G2_structured_cross_snapshot_reference	crossref
document_11174028_table_009.png	G1	hierarchy
document_11174028_table_009.png	G2	crossref
document_11332543_table_018.png	G1	hierarchy
document_11696987_table_005.png	G1	scoped
document_15151160_table_002.png	gap_001_column_spanner_variable_membership	hierarchy
document_437268_table_003.png	gap_001_hierarchical_column_header_relationships	hierarchy
document_437731_table_002.png	gap_001_variable_level_provenance_and_method_note_linkage	scoped
document_8886604_table_000.png	G1_variable_row_grouping	hierarchy
document_9089743_table_007.png	gap_001_aggregate_total_category_relationship	covered
002_BOSIB-ca473522-8ad0-4c80-9f0d-88bf887f2a2f_figure_000.png	G1	covered
120_PAD1205-ARABIC-PAD-PP152646-PUBLIC-Box393206B_figure_000.png	G1	panel
120_PAD1205-ARABIC-PAD-PP152646-PUBLIC-Box393206B_figure_001.png	gap_001_map_legend_layer_encoding_relationships	visual
161_28046_figure_000.png	G1_panel_level_relationships_in_composite_snapshot	panel
189_multi-page_figure_001.png	gap_001_panel_specific_metadata_bindings	panel
189_multi-page_figure_001.png	gap_002_multilevel_indicator_category_hierarchy	hierarchy
189_multi-page_figure_002.png	G1	visual
189_multi-page_figure_002.png	G2	cartography
191_multi-page_figure_002.png	gap_001_panel_scoped_metadata_bindings	panel
194_multi-page_figure_001.png	G1_panel_specific_metadata_binding	panel
194_multi-page_figure_001.png	G2_multilevel_table_row_hierarchy	hierarchy
195_multi-page_figure_001.png	gap_001_composite_panel_relationships	panel
196_multi-page_figure_001.png	G1	panel
196_multi-page_figure_001.png	G2	hierarchy
197_multi-page_figure_000.png	G1_nested_indicator_category_hierarchy	hierarchy
197_multi-page_figure_000.png	G2_panel_specific_metadata_association	panel
197_multi-page_figure_001.png	gap_001_panel_level_relationships_in_composite_snapshot	panel
197_multi-page_figure_002.png	G1_geospatial_coordinate_extent	cartography
197_multi-page_figure_002.png	G2_cartographic_scale_bar	cartography
197_multi-page_figure_002.png	G3_inset_map_panel_relationship	panel
199_multi0page_figure_000.png	G1	panel
199_multi0page_figure_000.png	G2	hierarchy
199_multi0page_figure_003.png	gap_001_cartographic_legend_symbol_feature_mapping	visual
200_multi-page_figure_003.png	gap_001_cartographic_scale_bar	cartography
202_multi0page_figure_001.png	G1	panel
202_multi0page_figure_002.png	G1_cartographic_scale_and_spatial_reference	cartography
202_multi0page_figure_002.png	G2_snapshot_map_identifier	covered
203_multi-page_figure_000.png	G1	panel
203_multi-page_figure_001.png	component_scoped_metadata_001	panel
203_multi-page_figure_002.png	G1	cartography
001_BOSIB-3f2311b3-9a20-44d3-b637-b3b2b3d21695_table_012.png	component_indicator_grouping	scoped
004_BOSIB-87c444de-4797-4bf9-b654-4932a7fb0112_table_004.png	hierarchical_pdo_outcome_indicator_structure	hierarchy
005_BOSIB-8191b179-7209-4faa-b5e0-11783bcd492d_table_002.png	gap_001_multilevel_financing_category_hierarchy	hierarchy
010_BOSIB1554c314c0a2187c019d7e85bc2a91_table_006.png	G1	covered
010_BOSIB1554c314c0a2187c019d7e85bc2a91_table_006.png	G2	operational
020_P1781250bdd2b50b0b9720d5c17632331c_table_001.png	G1	hierarchy
020_P1781250bdd2b50b0b9720d5c17632331c_table_001.png	G2	panel
020_P1781250bdd2b50b0b9720d5c17632331c_table_005.png	G1	operational
040_Iraq-COVID-19-Vaccination-Project_table_003.png	gap_001_nested_financing_source_hierarchy	hierarchy
040_Iraq-COVID-19-Vaccination-Project_table_003.png	gap_002_section_specific_variable_dimension_bindings	panel
055_Chad-COVID-19-Response-Project_table_002.png	G1	panel
193_multi-page_table_002.png	gap_001_multilevel_component_hierarchy	hierarchy
20200113_cimp_thematic_02_hudaydah_ceasefire_figure_003.png	gap_001_panel_specific_temporal_coverage	panel
66890_figure_005.png	G1	covered
cwc_survey_report_final_figure_003.png	G1	insufficient
finalreferralcarelebanon2014imcfinancial_figure_020.png	gap_001_hierarchical_financial_flow_relationships	operational
jordanreferralsataglance-2016_figure_012.png	G1	visual
oct_2023_rbsa_population_data_analysis_figure_002.png	G1_panel_level_metadata_relationships	panel
oct_2023_rbsa_population_data_analysis_figure_002.png	G2_directed_cross_border_flow_relationships	covered
oct_2023_rbsa_population_data_analysis_figure_008.png	G1_top_n_dimension_selection	covered
protection_trends_paper_no_7_jan-mar_2016_final_figure_004.png	G1_composite_panel_scoped_metadata	panel
protection_trends_paper_no_7_jan-mar_2016_final_figure_004.png	G2_visual_encoding_legend_mapping	visual
pswg_annual_report_2016_figure_009.png	G1_panel_component_bindings	panel
rapport_du_monitoring_de_protection_de_la_region_de_diffa_au_niger_janvier_2022_figure_007.png	G1	insufficient
rapport_du_monitoring_de_protection_tahoua-tillaberi_juillet2022_ciaud-antd_figure_002.png	G1_panel_level_semantic_associations	panel
tcd_hcr_mouvements_mixtes_update_janv-mars_2023_figure_001.png	gap_001_chart_dimension_encoding_roles	visual
unhcr_global_report_2020_-_west_and_central_africa_figure_005.png	gap_001	scoped
190415_gbv_secondary-data-analysis_cyclone-idai_moz_final_table_001.png	gap_001_variable_specific_provenance_dates_and_derivation	scoped
202501_bn_older_refugees_moldova_eng_table_000.png	G1	visual
points_saillants_situation_de_protection_en_rdc_apercu_semestriel_janvier-juin_2024_table_003.png	gap_001_total_categories_as_aggregates	covered
points_saillants_situation_de_protection_rdc_avril_2024_240523_table_000.png	geo-hierarchy-province-territory-001	covered
rapport_de_la_formation_sur_les_questions_transversales_pour_le_groupe_des_charges_de_communication_et_de_plaidoyer_humanitaire_de_lehp_table_002.png	G1_response_scale_applicability_by_question_block	scoped
rbsa_population_data_analysis_202304_1_table_000.png	G1_scoped_temporal_and_coverage_caveat	scoped
wfp-0000151334_table_001.png	G1_variable_scoped_footnote	scoped
wfp272177_table_000.png	gap_001	scoped
white_paper_social_media_3_0_table_000.png	G1_multiple_row_specific_temporal_coverage	scoped
zone_frontalire_nord_kivu_-_ituri_rpublique_dmocratique_du_congo_-_protection_analysis_update_juin_2022_table_004.png	G1_product_specific_units	covered
zone_frontalire_nord_kivu_-_ituri_rpublique_dmocratique_du_congo_-_protection_analysis_update_juin_2022_table_004.png	G2_temporal_crisis_status_column_mapping	scoped"""
FAMILY_BY_CANDIDATE_KEY = {}
for line in ADJUDICATION_TSV.strip().splitlines():
    snapshot_file_name, gap_id, family = line.split("\t")
    key = (snapshot_file_name, gap_id)
    if key in FAMILY_BY_CANDIDATE_KEY:
        raise ValueError(f"Duplicate adjudication key: {key}")
    FAMILY_BY_CANDIDATE_KEY[key] = family

observed_keys = {(row.snapshot_file_name, row.gap_id) for row in gaps_df.itertuples()}
unmapped = observed_keys - FAMILY_BY_CANDIDATE_KEY.keys()
stale = FAMILY_BY_CANDIDATE_KEY.keys() - observed_keys
if unmapped or stale:
    raise ValueError(f"Adjudication mismatch. Unmapped={unmapped}; stale={stale}")

review_df = gaps_for_display.copy()
review_df["family"] = [
    FAMILY_BY_CANDIDATE_KEY[(row.snapshot_file_name, row.gap_id)]
    for row in review_df.itertuples()
]
review_df["analyst_disposition"] = review_df["family"].map(
    lambda family: FAMILY_REVIEW[family][0]
)
review_df["requires_new_field"] = False
review_df["analyst_rationale"] = review_df["family"].map(
    lambda family: FAMILY_REVIEW[family][1]
)


In [ ]:
family_summary = (
    review_df.groupby(["family", "analyst_disposition"], as_index=False)
    .agg(
        candidate_rows=("gap_id", "size"),
        affected_snapshots=("snapshot_file_name", "nunique"),
    )
    .sort_values(["candidate_rows", "family"], ascending=[False, True])
)
retained = review_df[review_df["analyst_disposition"] == "retain_representation_issue"]
display(family_summary.style.hide(axis="index"))
display(
    Markdown(
        f"""The review retains **{len(retained)} representation rows** across
**{retained["snapshot_file_name"].nunique()} snapshots**, resolves or rejects
**{len(review_df) - len(retained)} rows**, and accepts **0 new fields**."""
    )
)


In [ ]:
display(
    review_df[
        [
            "snapshot_file_name",
            "source",
            "artifact_type",
            "gap_id",
            "gap_status",
            "missing_or_inadequately_represented_metadata",
            "closest_schema_paths",
            "family",
            "analyst_disposition",
            "analyst_rationale",
        ]
    ]
    .style.set_properties(**{"white-space": "pre-wrap", "text-align": "left"})
    .hide(axis="index")
)


### Interpretation

The substantive result is not a new inventory of missing fields. In this
first-pass review, none of the 90 surfaced candidates requires a new metadata
field. The evidence separates into:

- **26 panel/component-association rows**: the known pending limitation.
- **16 hierarchy rows**: one-level dimensions[].category_groups addresses the
  earlier grouping issue, but not recursive or deeper hierarchy.
- **12 other scoped-relationship rows**: values can be retained, but their
  attachment to a variable, dimension, subset, or component can be lost.
- **36 rejected or resolved rows**: covered, outside visual or cartographic
  scope, document-level cross-references, operational content, or insufficient
  evidence.

For this corpus, v1.2 therefore appears stable at the field/concept-inventory
level while remaining incomplete for some relational representation. This
refines the Validation 2 result: a one-level hierarchy representation was added,
but deeper hierarchies and broader scoping relationships remain. Panel-specific
associations also remain pending.

The conclusion is bounded to current v1.2, this corpus, this instrument, and the
surfaced candidates. It does not audit no-gap results, estimate recall, establish
universal completeness, or show that Pydantic caused or prevented a gap.


## Calibration controls

Calibration results are excluded from substantive counts. Calibration0 used the
complete schema. Calibration1 removed provenance and interpretive_notes; the
pre-specified signal was the displayed CIDTc source.


In [ ]:
calibration_rows = []
for run_name, path in CALIBRATION_PATHS.items():
    run_results = load_jsonl(path)
    run_errors = load_jsonl(CALIBRATION_ERROR_PATHS[run_name])
    successful = {row["snapshot_file_name"] for row in run_results}
    unresolved = [
        error
        for error in run_errors
        if error.get("snapshot_file_name") not in successful
    ]
    assessments = Counter(
        row["parsed_output"]["coverage_assessment"] for row in run_results
    )
    gaps = [
        gap
        for row in run_results
        for gap in row["parsed_output"]["critical_or_possible_gaps"]
    ]
    calibration_rows.append(
        {
            "run": run_name,
            "calls": len(run_results),
            "no_critical_gap_found": assessments["no_critical_gap_found"],
            "possible_gap": assessments["possible_gap"],
            "critical_gap_found": assessments["critical_gap_found"],
            "candidate_rows": len(gaps),
            "logged_error_attempts": len(run_errors),
            "unresolved_errors": len(unresolved),
        }
    )
display(pd.DataFrame(calibration_rows).style.hide(axis="index"))


**Control interpretation.** Calibration0 completed all 12 snapshots. Its five
logged error attempts came from the corrected nested-path validation bug; every
affected filename subsequently succeeded, so none is unresolved.

The provenance ablation returned a critical candidate for the displayed CIDTc
data source and a possible construction-note candidate. The pre-specified
sensitivity signal passed. The paired complete-schema result returned no
critical gap. This shows sensitivity to the removed capacity, not general recall.


## Candidate inspector and full overview


In [ ]:
def inspect_candidate(snapshot_file_name: str) -> None:
    """Display one snapshot and its adjudicated candidates.

    Parameters
    ----------
    snapshot_file_name : str
        Successful filename with at least one candidate.

    Raises
    ------
    KeyError
        If no candidate exists for the filename.
    ValueError
        If the image cannot be resolved uniquely.
    """
    selected = review_df[review_df["snapshot_file_name"] == snapshot_file_name]
    if selected.empty:
        raise KeyError(f"No candidates for {snapshot_file_name!r}.")
    matches = list(SNAPSHOTS_DIR.rglob(snapshot_file_name))
    if len(matches) != 1:
        raise ValueError(f"Expected one image; found {len(matches)}.")
    display(Markdown(f"### {snapshot_file_name}"))
    display(Image(filename=str(matches[0]), width=1000))
    display(
        selected.style.set_properties(
            **{"white-space": "pre-wrap", "text-align": "left"}
        ).hide(axis="index")
    )


candidate_snapshots = review_df.groupby(
    ["snapshot_file_name", "source", "artifact_type"], as_index=False
).agg(
    candidate_rows=("gap_id", "size"),
    families=("family", lambda values: ", ".join(sorted(set(values)))),
)
display(candidate_snapshots.style.hide(axis="index"))
SNAPSHOT_FILE_NAME = candidate_snapshots.iloc[0]["snapshot_file_name"]
inspect_candidate(SNAPSHOT_FILE_NAME)


In [ ]:
display(
    results_df[
        [
            "snapshot_file_name",
            "source",
            "artifact_type",
            "coverage_assessment",
            "candidate_gaps",
            "elapsed_seconds",
            "input_tokens",
            "cached_input_tokens",
            "output_tokens",
            "reasoning_tokens",
            "cost_usd",
        ]
    ]
    .sort_values(["source", "artifact_type", "snapshot_file_name"])
    .style.format(
        {
            "elapsed_seconds": "{:,.1f}",
            "input_tokens": "{:,}",
            "cached_input_tokens": "{:,}",
            "output_tokens": "{:,}",
            "reasoning_tokens": "{:,}",
            "cost_usd": "$ {:,.4f}",
        }
    )
    .hide(axis="index")
)


## Unresolved errors


In [ ]:
if unresolved_errors:
    display(pd.DataFrame(unresolved_errors))
else:
    display(Markdown("_No unresolved full-run errors._"))
